# Batch 5: long rollout (6 s), PINC vs plain NN, 1-DOF pitch with motors off (impulse)
**PINC** = a network that takes the state now and the input, and gives the state a short time later:

PINC(theta, q, M, dt) -> (theta, q) after dt

Chaining it (output becomes the next input) simulates a whole flight for any input M.

Trained with one loss, three parts:
1. **data**: from each measured state, one step of DT = 0.02 s must land on the next measured state
2. **start**: with dt = 0 the output must equal the input state
3. **physics**: at random made-up states, inputs and dt, the output must obey the hover equation
   theta' = q,  q' = -2 xi wn q - wn^2 (theta - th0) + mu M, with the constants the simulation fit finds on the train flights (best model on impulse, VAF 97 %)

**Plain NN** = exactly the same network and training, with the physics part switched off.

Why impulse: the hover flights only last about 1.5 s before the drone tips over, so the rollout cannot be longer than that. The impulse flights swing for 6 s, so here every model must predict 6 s ahead (300 steps of 0.02 s) with no correction.

Train: impulse_1 to impulse_4. Test: impulse_5 (unseen). 5 seeds per model.

## 0. Setup

In [ ]:
import numpy as np, pandas as pd, torch, time, matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
torch.set_num_threads(4)
D = '../data/1dof/'
FL = pd.read_csv(D + 'flights_1dof.csv').set_index('file')
TAU = 0.02                     # delay command -> motors [s]
DT = 0.02                      # one PINC step [s] (5 samples at 250 Hz)
TRAIN = [f'impulse_{i}.csv' for i in range(1, 5)]
TEST = 'impulse_5.csv'
WIN = 6.0                      # window length [s]: 0.1 s before the push, then 6 s of swinging
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 20000
print('train', TRAIN, 'test', TEST)

## 1. Data: each flight (6 s) on a 0.02 s grid; M = delayed motor command averaged over the step

In [ ]:
def load(name):
    r = FL.loc[name]; df = pd.read_csv(D + name)
    f = df[(df.t >= r.inj_start_s - 0.1 - TAU) & (df.t < r.inj_start_s - 0.1 + WIN)].reset_index(drop=True)
    f['t'] = f.t - f.t[0] - TAU                                    # t = 0 at the start; keep 0.02 s before it for the delay
    return f

def steps(f):
    t = np.arange(0, f.t.iloc[-1] - DT, DT)
    th = np.interp(t, f.t, f.theta); q = np.interp(t, f.t, f.q)
    M = np.array([np.mean(np.interp(np.linspace(ti, ti + DT, 6)[:-1] - TAU, f.t, f.M)) for ti in t])
    return t, th, q, M

flights = {n: load(n) for n in TRAIN + [TEST]}
grid = {n: steps(f) for n, f in flights.items()}
print('training pairs:', sum(len(g[0]) - 1 for n, g in grid.items() if n in TRAIN), '| test steps:', len(grid[TEST][0]))

## 2. Physics, written once

In [ ]:
def accel(theta, q, M, p):
    return -2 * p['xi'] * p['wn'] * q - p['wn']**2 * (theta - p['th0']) + p['mu'] * M

def euler_like_ivp(p):
    # equation model as a step function: exact ODE solve over one DT with M held constant
    def step(th, q, M):
        y = solve_ivp(lambda t, z: [z[1], accel(z[0], z[1], M, p)], (0, DT), [th, q], max_step=DT / 5).y[:, -1]
        return y[0], y[1]
    return step

def vaf(y, yhat):
    return max(1 - np.var(y - yhat) / np.var(y), 0) * 100

## 2b. Simulation fit on the train flights (gives the constants for the physics loss and the equation baseline)

In [ ]:
def run_ivp(p, f):
    t, M = f.t.values, f.M.values
    rhs = lambda tt, z: [z[1], accel(z[0], z[1], np.interp(tt - TAU, t, M), p)]
    i0 = np.searchsorted(t, 0.0)
    return solve_ivp(rhs, (t[i0], t[-1]), [f.theta[i0], f.q[i0]], t_eval=t[i0:], max_step=0.004).y[0], i0
keys = ['wn', 'xi', 'mu', 'th0']
def residual(v):
    p = dict(zip(keys, v))
    out = []
    for n in TRAIN:
        y, i0 = run_ivp(p, flights[n]); out.append(y - flights[n].theta.values[i0:])
    return np.concatenate(out)
fit = least_squares(residual, [4.0, 0.2, 80, 0], bounds=([0.5, 0, 0, -0.3], [20, 2, 2000, 0.3]))
P_SIM = dict(zip(keys, fit.x)); P_PINN = P_SIM          # PINC physics uses the sim-fit constants
print('sim fit constants:', {k: round(v, 4) for k, v in P_SIM.items()})

## 3. The network and the PINC step

In [ ]:
def mlp(n_in, n_out, width=32, depth=3):
    layers, n = [], n_in
    for _ in range(depth):
        layers += [torch.nn.Linear(n, width), torch.nn.Tanh()]; n = width
    return torch.nn.Sequential(*layers, torch.nn.Linear(n, n_out))

col = lambda x: torch.tensor(np.asarray(x), dtype=torch.float32).reshape(-1, 1)
S = dict(th=0.3, q=4.0, M=0.1)          # normalisation scales (inputs and outputs about -1..1)

def pinc_model(net):
    def step(th, q, M, dt):              # all torch columns; returns (theta, q) after dt
        x = torch.cat([th / S['th'], q / S['q'], M / S['M'], dt / DT], 1)
        o = net(x)
        return S['th'] * o[:, :1], S['q'] * o[:, 1:2]
    return step

## 4. Training function (PINC when w_phys = 1, plain NN when w_phys = 0)

In [ ]:
def train_pinc(names, seed=0, epochs=EPOCHS, lr=1e-3, w_phys=1.0, n_col=2000, p=P_PINN):
    # data
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    G = [grid[n] for n in names]
    th0 = col(np.concatenate([g[1][:-1] for g in G])); q0 = col(np.concatenate([g[2][:-1] for g in G]))
    th1 = col(np.concatenate([g[1][1:] for g in G])); q1 = col(np.concatenate([g[2][1:] for g in G]))
    M0 = col(np.concatenate([g[3][:-1] for g in G]))
    # network
    net = mlp(4, 2); step = pinc_model(net)
    # optimizer
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=lr / 20)
    # loop
    hist = []
    for e in range(epochs):
        a, b = step(th0, q0, M0, torch.full_like(th0, DT))                    # one step of DT
        data = (((a - th1) / S['th'])**2).mean() + (((b - q1) / S['q'])**2).mean()
        a, b = step(th0, q0, M0, torch.zeros_like(th0))                       # dt = 0 -> same state
        start = (((a - th0) / S['th'])**2).mean() + (((b - q0) / S['q'])**2).mean()
        physics = torch.tensor(0.0)
        if w_phys > 0:
            zt = col(rng.uniform(-0.35, 0.35, n_col)); zq = col(rng.uniform(-4, 4, n_col))
            zM = col(rng.uniform(-0.02, 0.12, n_col)); dt = col(rng.uniform(0, DT, n_col)).requires_grad_(True)
            a, b = step(zt, zq, zM, dt)
            da = torch.autograd.grad(a, dt, torch.ones_like(a), create_graph=True)[0]    # d theta / d dt
            db = torch.autograd.grad(b, dt, torch.ones_like(b), create_graph=True)[0]    # d q / d dt
            physics = (((da - b) / S['q'])**2).mean() + (((db - accel(a, b, zM, p)) / (S['q'] / DT))**2).mean()
            a, b = step(zt, zq, zM, torch.zeros_like(zt))
            start = start + (((a - zt) / S['th'])**2).mean() + (((b - zq) / S['q'])**2).mean()
        loss = data + start + w_phys * physics
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if e % 1000 == 0 or e == epochs - 1:
            hist.append((e, data.item(), start.item(), physics.item()))
    return dict(net=net, step=step, history=pd.DataFrame(hist, columns=['epoch', 'data', 'start', 'physics']))

## 5. Runners: whole-flight rollout and one-step-ahead

In [ ]:
def run_open(step_fn, th_start, q_start, M_list, is_torch=True):
    # chain the model: each output is the next input
    th, q = th_start, q_start; out = [th]
    for M in M_list[:-1]:
        if is_torch:
            with torch.no_grad():
                a, b = step_fn(col([th]), col([q]), col([M]), col([DT])); th, q = a.item(), b.item()
        else:
            th, q = step_fn(th, q, M)
        out.append(th)
    return np.array(out)

def run_one_step(step_fn, g, is_torch=True):
    t, th, q, M = g; out = [th[0]]
    for k in range(len(t) - 1):
        if is_torch:
            with torch.no_grad(): a, _ = step_fn(col([th[k]]), col([q[k]]), col([M[k]]), col([DT])); out.append(a.item())
        else:
            out.append(step_fn(th[k], q[k], M[k])[0])
    return np.array(out)

## 6. Train: 5 seeds x (PINC, plain NN)

In [ ]:
runs = {'PINC': [], 'plain NN': []}
for s in SEEDS:
    for label, wp in [('PINC', 1.0), ('plain NN', 0.0)]:
        t0 = time.time(); r = train_pinc(TRAIN, seed=s, w_phys=wp); r['time'] = time.time() - t0
        runs[label].append(r); print(label, 'seed', s, f"{r['time']:.0f} s", r['history'].iloc[-1][['data', 'start', 'physics']].round(7).to_dict())

## 7. Test on impulse_5 (unseen): 6 s rollout, error vs horizon

In [ ]:
g = grid[TEST]; t, th, q, M = g
HOR = [1.0, 2.0, 4.0, 5.9]                              # seconds after the start
idx = [int(round(h / DT)) for h in HOR]
def rmse_upto(pred, h):                                   # RMSE over the first h seconds of the rollout
    n = int(round(h / DT)) + 1; return np.degrees(np.sqrt(np.mean((pred[:n] - th[:n])**2)))
def score(pred_roll, pred_one):
    return [vaf(th, pred_roll), np.degrees(np.sqrt(np.mean((pred_one - th)**2)))] + [rmse_upto(pred_roll, h) for h in HOR]
cols = ['rollout VAF [%]', 'one-step RMSE [deg]'] + [f'rollout RMSE first {h} s [deg]' for h in HOR]
rows = {}
for label, p in [('Equation, sim-fit constants', P_SIM), ('Equation, ANT-X published constants', dict(wn=3.437, xi=0.0836, mu=358.5, th0=P_SIM['th0']))]:
    st = euler_like_ivp(p)
    rows[label] = score(run_open(st, th[0], q[0], M, False), run_one_step(st, g, False))
preds = {}
for label, rs in runs.items():
    S_ = np.array([score(run_open(r['step'], th[0], q[0], M), run_one_step(r['step'], g)) for r in rs])
    preds[label] = [run_open(r['step'], th[0], q[0], M) for r in rs]
    rows[label + ' (mean of 5)'] = S_.mean(0); rows[label + ' (std)'] = S_.std(0)
table = pd.DataFrame(rows, index=cols).T.round(3)
table.to_csv('../results/batch5_long_rollout.csv')
table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(t, np.degrees(th), 'k', lw=2, label='measured impulse_5')
ax[0].plot(t, np.degrees(run_open(euler_like_ivp(P_SIM), th[0], q[0], M, False)), ':', label='equation, sim-fit constants')
for label, ls in [('PINC', '-'), ('plain NN', '--')]:
    for i, y in enumerate(preds[label]):
        ax[0].plot(t, np.degrees(y), ls, color='C0' if label == 'PINC' else 'C3', alpha=0.6, label=label + ' (5 seeds)' if i == 0 else None)
ax[0].set(xlabel='t [s]', ylabel='theta [deg]', title='Rollout on the unseen flight'); ax[0].legend(fontsize=8)
for label, color in [('PINC', 'C0'), ('plain NN', 'C3')]:
    E = np.array([np.degrees(np.abs(y - th)) for y in preds[label]])
    ax[1].plot(t, E.mean(0), color=color, label=label); ax[1].fill_between(t, E.min(0), E.max(0), color=color, alpha=0.2)
ax[1].set(xlabel='t [s]', ylabel='|theta error| [deg]', title='Error vs horizon (band = 5 seeds)'); ax[1].legend()
plt.tight_layout(); plt.savefig('../figures/batch5_long_rollout.png', dpi=120, bbox_inches='tight'); plt.show()

## 8. What this shows
- **Rollout RMSE over the first 1, 2, 4, 5.9 s:** how the error grows as the model predicts further ahead with no correction.
- A model that drifts will have a fast-growing number; a model that learned the physics should keep its error flat after the swing dies out.